# This script is used to apply arcgis-python API

In [1]:
# ---------------------------------------------------------
# the liner habitat, wadi and gullies from phytosociological dataset contains point records representing linear habitat patches. 
# Each record includes attributes:
# Wide_M → estimated habitat width
# Length_M → estimated habitat length
# Exposure → approximate orientation (e.g., N, NE, SW)
# The goal is to convert these points into rectangular habitat polygons representing the approximate spatial footprint of each linear habitat patch.
# Key logic:
# Rectangle center = survey point
# Rectangle width = Wide_M
# Rectangle length = Length_M
# Rectangle rotation = Exposure direction
# ---------------------------------------------------------

In [2]:
import os
import geopandas as gpd
from shapely.geometry import box
from shapely.affinity import rotate

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
input_dir = r"C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Habitat mapping"
input_path = os.path.join(input_dir,"Phytosociologicalsurvey_Areasforlinearhabitats.shp")
output_path = os.path.join(input_dir,"Phytosociologicalsurvey_linearhabitats_rect.shp")

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
gdf = gpd.read_file(input_path)

print("Input CRS:", gdf.crs)

# ---------------------------------------------------------
# Ensure projected CRS (meters)
# Bouarfa region → UTM 30N
# ---------------------------------------------------------
if gdf.crs.is_geographic:
    gdf = gdf.to_crs(epsg=32630)

# ---------------------------------------------------------
# Valid exposure directions
# ---------------------------------------------------------
valid_dirs = ['N','S','E','W','NE','NW','SE','SW']

# ---------------------------------------------------------
# Direction → rotation angle (degrees)
# ---------------------------------------------------------
dir_to_angle = {
    'N':90,
    'S':270,
    'E':0,
    'W':180,
    'NE':45,
    'NW':135,
    'SE':315,
    'SW':225
}

# ---------------------------------------------------------
# Function to create rectangle
# ---------------------------------------------------------
def create_rectangle(row):

    point = row.geometry
    x, y = point.x, point.y

    width = row["Wide_M"]
    length = row["Length_M"]

    if width is None or length is None:
        return None

    half_w = width / 2
    half_l = length / 2

    # Create axis-aligned rectangle
    rect = box(
        x - half_l,
        y - half_w,
        x + half_l,
        y + half_w
    )

    exposure = str(row["Exposure"]).upper()

    # Rotate only if valid direction
    if exposure in valid_dirs:

        angle = dir_to_angle[exposure]

        rect = rotate(
            rect,
            angle,
            origin=(x,y),
            use_radians=False
        )

    return rect

# ---------------------------------------------------------
# Generate polygons
# ---------------------------------------------------------
gdf["geometry"] = gdf.apply(create_rectangle, axis=1)

# Remove null geometries
gdf = gdf[gdf.geometry.notnull()]

# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------
gdf.to_file(output_path)

print("Output saved to:")
print(output_path)

Input CRS: EPSG:4326
Output saved to:
C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Habitat mapping\Phytosociologicalsurvey_linearhabitats_rect.shp


In [1]:
import arcpy
import os

# Input and output paths
input_dir = r"C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Reneco/Vegetation Cover and Food Availability Studies"
input_points = os.path.join(input_dir, "Trophic_survey_data_114_station_habitat.shp")

# Temporary projected file
projected_points = os.path.join(input_dir, "Trophic_survey_data_114_station_habitat_UTM30N.shp")

# Final output polygons
output_polygons = os.path.join(input_dir, "Trophic_survey_data_114_station_habitat_polygon.shp")

ModuleNotFoundError: No module named 'arcpy'

In [5]:
# Target spatial reference (WGS 1984 UTM Zone 30N)
utm_30n = arcpy.SpatialReference(32630)

# Step 1: Project input points into UTM 30N
arcpy.management.Project(
    in_dataset=input_points,
    out_dataset=projected_points,
    out_coor_system=utm_30n
)

# Step 2: Get fields from input shapefile (exclude geometry + OID)
fields = [f.name for f in arcpy.ListFields(projected_points) 
          if f.type not in ("Geometry", "OID")]

In [6]:
# Step 3: Create a new, empty polygon feature class with same attributes
# Set side lengths (meters)
width = 15
height = 28

if arcpy.Exists(output_polygons):
    arcpy.Delete_management(output_polygons)

arcpy.CreateFeatureclass_management(
    out_path=os.path.dirname(output_polygons),
    out_name=os.path.basename(output_polygons),
    geometry_type="POLYGON",
    spatial_reference=utm_30n
)

# Add same fields to output polygons
for field in fields:
    f = arcpy.ListFields(projected_points, field)[0]
    if f.type not in ("OID", "Geometry"):
        arcpy.AddField_management(output_polygons, field, f.type, 
                                  f.precision, f.scale, f.length, 
                                  f.aliasName, f.isNullable, f.required)

# Step 4: Copy geometry + attributes
insert_fields = ["SHAPE@"] + fields
search_fields = ["SHAPE@XY"] + fields

with arcpy.da.InsertCursor(output_polygons, insert_fields) as icursor:
    with arcpy.da.SearchCursor(projected_points, search_fields) as scursor:
        for row in scursor:
            x, y = row[0]

            # Rectangle corners
            lower_left_x = x - (width / 2)
            lower_left_y = y - (height / 2)
            upper_right_x = x + (width / 2)
            upper_right_y = y + (height / 2)

            # Polygon geometry
            array = arcpy.Array([
                arcpy.Point(lower_left_x, lower_left_y),
                arcpy.Point(upper_right_x, lower_left_y),
                arcpy.Point(upper_right_x, upper_right_y),
                arcpy.Point(lower_left_x, upper_right_y),
                arcpy.Point(lower_left_x, lower_left_y)  # close the ring
            ])
            polygon = arcpy.Polygon(array, utm_30n)

            # Insert polygon + attributes (skip XY)
            icursor.insertRow([polygon] + list(row[1:]))

print("Rectangular polygons with attributes created successfully in:", output_polygons)

✅ Rectangular polygons with attributes created successfully in: C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Reneco/Vegetation Cover and Food Availability Studies\Trophic_survey_data_114_station_habitat_polygon.shp


In [4]:
# Add 'Habvtat_id' field to the station polygons, based on unique char values in 'Habitat'
# Path to your polygon shapefile
fc = r"C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Reneco/Vegetation Cover and Food Availability Studies/Trophic_survey_data_114_station_habitat_polygon.shp"

# Step 1: Collect unique Habitat values
habitat_values = set()
with arcpy.da.SearchCursor(fc, ["Habitat"]) as cursor:
    for row in cursor:
        if row[0] is not None:
            habitat_values.add(row[0])

# Step 2: Create a mapping {Habitat_name : ID}
habitat_mapping = {val: i+1 for i, val in enumerate(sorted(habitat_values))}
print("Habitat mapping:", habitat_mapping)

# Step 3: Add new field Habitat_id (if not exists)
field_names = [f.name for f in arcpy.ListFields(fc)]
if "Habitat_id" not in field_names:
    arcpy.AddField_management(fc, "Habitat_id", "LONG")

# Step 4: Update Habitat_id using the mapping
with arcpy.da.UpdateCursor(fc, ["Habitat", "Habitat_id"]) as cursor:
    for row in cursor:
        if row[0] in habitat_mapping:
            row[1] = habitat_mapping[row[0]]
            cursor.updateRow(row)

print("Habitat_id field created and populated successfully")

Habitat mapping: {'Alfah-grass steppe': 1, 'Drainage lines': 2, 'Flooded wadi': 3, 'Reg': 4, 'Salt marshes': 5, 'Wadi': 6}
✅ Habitat_id field created and populated successfully


In [ ]:
import arcpy

# Paths to your training polygon shapefiles
TRAIN_TABLES = {
    "100m": r"C:/path/to/AI_Baten_rdpoints_100mbuf_train.shp",
    "300m": r"C:/path/to/AI_Baten_rdpoints_300mbuf_train.shp",
    "500m": r"C:/path/to/AI_Baten_rdpoints_500mbuf_train.shp"
}

C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Habitat mapping/

for name, fc in TRAIN_TABLES.items():
    print(f"Processing {name} dataset...")
    
    # Step 1: Collect unique Habitat values
    habitat_values = set()
    with arcpy.da.SearchCursor(fc, ["Habitat"]) as cursor:
        for row in cursor:
            if row[0] is not None:
                habitat_values.add(row[0])
    
    # Step 2: Create mapping {Habitat_name : ID}
    habitat_mapping = {val: i+1 for i, val in enumerate(sorted(habitat_values))}
    print(f"Habitat mapping for {name}:", habitat_mapping)
    
    # Step 3: Add new field Habitat_id (if not exists)
    field_names = [f.name for f in arcpy.ListFields(fc)]
    if "Habitat_id" not in field_names:
        arcpy.AddField_management(fc, "Habitat_id", "LONG")
    
    # Step 4: Update Habitat_id using the mapping
    with arcpy.da.UpdateCursor(fc, ["Habitat", "Habitat_id"]) as cursor:
        for row in cursor:
            if row[0] in habitat_mapping:
                row[1] = habitat_mapping[row[0]]
                cursor.updateRow(row)
    
    print(f"Habitat_id field created and populated for {name}\n")

In [4]:
import os
os.getcwd()

'C:\\Users\\PangY\\Python-GIS'

In [6]:
import numpy as np
import rasterio
from rasterio.mask import mask
import geopandas as gpd

# ----------------------------
# INPUT
# ----------------------------
# raster_path = "C:/Users/PangY/OneDrive - Smithsonian Institution/Bustard/01_Data/Habitat mapping/AlBaten_Composite_2024.tif"
raster_path = "AI_Baten_S2_202503.tif"
aoi_path = "Al_Baten_study_area.shp"

# ----------------------------
# LOAD AOI
# ----------------------------
aoi = gpd.read_file(aoi_path)

# Ensure CRS matches raster
with rasterio.open(raster_path) as src:
    if aoi.crs != src.crs:
        aoi = aoi.to_crs(src.crs)

    # Convert geometries to GeoJSON format
    geometries = [geom.__geo_interface__ for geom in aoi.geometry]

    # ----------------------------
    # CLIP / MASK
    # ----------------------------
    clipped, transform = mask(
        src,
        geometries,
        crop=True,        # crop to AOI extent
        nodata=np.nan     # optional
    )

    out_meta = src.meta.copy()

# ----------------------------
# UPDATE METADATA
# ----------------------------
out_meta.update({
    "height": clipped.shape[1],
    "width": clipped.shape[2],
    "transform": transform
})

# ----------------------------
# SAVE OUTPUT
# ----------------------------
out_path = "AI_Baten_S2_202503_clipped.tif"

with rasterio.open(out_path, "w", **out_meta) as dest:
    dest.write(clipped)

print("Saved clipped raster:", out_path)

Saved clipped raster: AI_Baten_S2_202503_clipped.tif
